<a href="https://colab.research.google.com/github/Innovatewithapple/BioasqRAG/blob/main/BioASQRAG%CC%88Dense.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import random
import os
import torch

def setProductionSeed(isActive,seed=42):
  if isActive == True:
    random.seed(seed)
    np.random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
      torch.cuda.manual_seed(seed)
      torch.cuda.manual_seed_all(seed)

      torch.backends.cudnn.deterministic = True
      torch.backends.cudnn.benchmark = False

setProductionSeed(True)

In [ ]:
!pip install -U bitsandbytes>=0.46.1

In [ ]:
!pip install rank-bm25

In [ ]:
!pip install transformers datasets

In [ ]:
import torch.nn as nn
from transformers import AutoModel,AutoTokenizer,AutoModelForCausalLM,AutoModelForSequenceClassification
from datasets import load_dataset
from torch.utils.data import DataLoader,Dataset
from tqdm import tqdm
import torch.nn.functional as F
import re
import nltk
nltk.download('punkt')
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt to /usr/share/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /usr/share/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [ ]:
from nltk.tokenize import sent_tokenize
import pandas as pd
from rank_bm25 import BM25Okapi
import gc
from transformers import BitsAndBytesConfig

In [ ]:
EvaluationData = {}

In [ ]:
text_corpus = load_dataset('enelpol/rag-mini-bioasq','text-corpus')
question_answer_passage = load_dataset('enelpol/rag-mini-bioasq','question-answer-passages')

README.md: 0.00B [00:00, ?B/s]

text-corpus/test-00000-of-00001.parquet:   0%|          | 0.00/35.3M [00:00<?, ?B/s]

Generating test split:   0%|          | 0/40181 [00:00<?, ? examples/s]

question-answer-passages/train-00000-of-(…):   0%|          | 0.00/1.12M [00:00<?, ?B/s]

question-answer-passages/test-00000-of-0(…):   0%|          | 0.00/187k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/4012 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/707 [00:00<?, ? examples/s]

In [ ]:
text_corpus

DatasetDict({
    test: Dataset({
        features: ['passage', 'id'],
        num_rows: 40181
    })
})

In [ ]:
corpus_data = text_corpus['test']
corpus_data

Dataset({
    features: ['passage', 'id'],
    num_rows: 40181
})

In [ ]:
all_sentences = []

for data in corpus_data:
    sentences = sent_tokenize(data['passage'])

    for sentence in sentences:
        all_sentences.append(sentence)

with open("all_sentences.txt", "w", encoding="utf-8") as f:
    for sentence in all_sentences:
        f.write(sentence + "\n\n")

In [ ]:
rows = []

for data in corpus_data:

    passage_id = data['id']

    sentences = sent_tokenize(data['passage'])

    for sentence_idx, sentence in enumerate(sentences):

        rows.append({
            'passage_id': passage_id,
            'sentence_id': sentence_idx,
            'sentence': sentence
        })

df = pd.DataFrame(rows)

df.to_csv(
    "sentence_analysis_with_ids.csv",
    index=False
)

In [ ]:
def ExtractDataAndCreateChunks(chunksize, overlapsize):
    chunks = []
    for data in corpus_data:
        # Split into sentences
        sentences = re.split(r'(?<=[.!?])\s+', data['passage'])

        current_chunk = []
        current_length = 0

        for sentence in sentences:

            sentence_words = sentence.split()
            sentence_length = len(sentence_words)

            # If adding sentence exceeds chunk size
            if current_length + sentence_length > chunksize:

                chunk_text = " ".join(current_chunk)

                chunks.append({
                    'original_passage_id': data['id'],
                    'chunk_text': chunk_text
                })

                # overlap handling
                overlap_sentences = current_chunk[-overlapsize:]

                current_chunk = overlap_sentences
                current_length = sum(len(sentence.split()) for sentence in current_chunk)

            current_chunk.append(sentence)
            current_length += sentence_length

        # Add remaining chunk
        if current_chunk:

            chunk_text = " ".join(current_chunk)

            chunks.append({
                'original_passage_id': data['id'],
                'chunk_text': chunk_text
            })

    return chunks
knowledge_strings = ExtractDataAndCreateChunks(chunksize=100,overlapsize=1)

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

device(type='cuda')

# **Encoder Model**

In [ ]:
# encoder_token_model = AutoTokenizer.from_pretrained('BAAI/bge-m3')
# encoder_token_model.vocab_size

In [ ]:
class RAGKnowledgeDataset(Dataset):
  def __init__(self,encoding):
    self.encoding = encoding

  def __len__(self):
    return len(self.encoding['input_ids'])

  def __getitem__(self,idx):
    input_ids = self.encoding['input_ids'][idx]
    attention_mask = self.encoding['attention_mask'][idx]

    return {
        'input_ids':input_ids,
        'attention_mask':attention_mask
    }

In [ ]:
# Model 1: for passages (replaces BGEWriter for database encoding)
class MedCPTPassageEncoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.model = AutoModel.from_pretrained('ncbi/MedCPT-Article-Encoder')

    def forward(self, input_ids, attention_mask):
        output = self.model(input_ids=input_ids, attention_mask=attention_mask)
        return output.last_hidden_state[:, 0, :]  # CLS token, not mean pooling

# Model 2: for queries only
class MedCPTQueryEncoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.model = AutoModel.from_pretrained('ncbi/MedCPT-Query-Encoder')

    def forward(self, input_ids, attention_mask):
        output = self.model(input_ids=input_ids, attention_mask=attention_mask)
        return output.last_hidden_state[:, 0, :]  # CLS token

In [ ]:
passage_encoder = MedCPTPassageEncoder().to(device)
passage_encoder.eval()

query_encoder = MedCPTQueryEncoder().to(device)
query_encoder.eval()

# Two separate tokenizers too
from transformers import AutoTokenizer
passage_tokenizer = AutoTokenizer.from_pretrained('ncbi/MedCPT-Article-Encoder')
query_tokenizer = AutoTokenizer.from_pretrained('ncbi/MedCPT-Query-Encoder')

config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/438M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/438M [00:00<?, ?B/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/74.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

In [ ]:
knowledge_text = [i['chunk_text'] for i in knowledge_strings]
encoder_tokenised_data = passage_tokenizer(text=knowledge_text,padding=True,max_length=256,add_special_tokens=True,truncation=True,return_attention_mask=True,return_tensors='pt')

In [ ]:
encoder_data = RAGKnowledgeDataset(encoder_tokenised_data)
encoder_Data_Loader = DataLoader(dataset=encoder_data,batch_size=32,shuffle=False,pin_memory=True,num_workers=2)

In [ ]:
# class BGEWriter(nn.Module):
#   def __init__(self) -> None:
#     super().__init__()
#     self.bge = AutoModel.from_pretrained('BAAI/bge-m3')

#   def forward(self,input_ids,attention_mask):
#     bgeOutput = self.bge(input_ids=input_ids,attention_mask=attention_mask)
#     bgeOutput = bgeOutput.last_hidden_state
#     mean = attention_mask.unsqueeze(-1).float()
#     mean = (bgeOutput*mean).sum(dim=1)/mean.sum(dim=1)
#     return mean

In [ ]:
# encoder_Model = BGEWriter().to(device)
# encoder_Model.eval()

In [ ]:
# database_vectors = []
# with torch.no_grad():
#   with torch.amp.autocast('cuda'):
#     for batch in tqdm(encoder_Data_Loader,desc='Encoding Model'):
#       input_ids,attention_mask = batch['input_ids'].to(device),batch['attention_mask'].to(device)

#       model_outputs = encoder_Model(input_ids,attention_mask)
#       database_vectors.append(model_outputs)

# vector_database = torch.cat(database_vectors,dim=0).cpu()
# print(vector_database.shape)

In [ ]:
gc.collect()
torch.cuda.empty_cache()

In [ ]:
database_vectors = []
with torch.no_grad():
    with torch.amp.autocast('cuda'):
        for batch in tqdm(encoder_Data_Loader, desc='Encoding passages'):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            output = passage_encoder(input_ids, attention_mask)  # passage encoder
            database_vectors.append(output.cpu())

vector_database = torch.cat(database_vectors, dim=0)
del output

gc.collect()
torch.cuda.empty_cache()

Encoding passages: 100%|██████████| 4077/4077 [07:35<00:00,  8.96it/s]


In [ ]:
train_quesAnsPassage = question_answer_passage['train']
train_quesAnsPassage

Dataset({
    features: ['question', 'answer', 'id', 'relevant_passage_ids'],
    num_rows: 4012
})

In [ ]:
Reranking_token_Model = AutoTokenizer.from_pretrained("ncbi/MedCPT-Cross-Encoder")
rerank_model = AutoModelForSequenceClassification.from_pretrained("ncbi/MedCPT-Cross-Encoder")
rerank_model.eval()

config.json:   0%|          | 0.00/741 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/74.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/228 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/438M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12,

In [ ]:

# del decoder_tokenizer
# gc.collect()
# torch.cuda.empty_cache()

In [ ]:
decoder_tokenizer = AutoTokenizer.from_pretrained('Qwen/Qwen2.5-3B-Instruct')
decoder_model = AutoModelForCausalLM.from_pretrained(
    'Qwen/Qwen2.5-3B-Instruct',
    dtype=torch.float16,
    device_map='auto'
)

model.safetensors:   0%|          | 0.00/438M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

In [ ]:
def GetFinalAnswer(input_ids_decoder,attention_mask):
  with torch.no_grad():
   with torch.amp.autocast('cuda'):
    output_ids = decoder_model.generate(
        input_ids_decoder,
        attention_mask=attention_mask,
        max_new_tokens=100,       # Gives it plenty of room to write a complete sentence [prev]
        do_sample=False,          # Enables natural conversational sampling [prev]
        top_k=10,                # Restricts the word pool to keep it highly focused [prev]
        temperature=0.2,         # Drops creativity near zero to lock onto context facts [prev]
        repetition_penalty=1.15, # Nudges it away from loops without breaking grammar rules [prev]
        no_repeat_ngram_size=0,  # Turned off: Disables the rigid, destructive phrase bans completely [prev]
        pad_token_id=decoder_tokenizer.eos_token_id
    )

   newTokens = output_ids[0,input_ids_decoder.shape[1]:]
   final_output = decoder_tokenizer.decode(newTokens,skip_special_token=True)
   print('FinalAnswer:', final_output)
   EvaluationData["predicted_answer"] = final_output
   return final_output


In [ ]:
# def ManageDecoderModel(finalChunk, question):
#     messages = [                          # indent this
#         {
#             "role": "system",
#             "content": """You are a biomedical expert answering questions based on scientific literature.

# Instructions:
# - Synthesize information from ALL provided context passages
# - Focus on specific biological mechanisms, markers, findings and specifically the names
# - Give a direct answer in 1-2 sentence mentioning all specific histone markers and mechanisms found across the context passages.
# - Use proper biomedical terminology
# - Find detailed information from the context starting to end
# - Do not add information beyond what is in the context"""
#         },
#         {
#             "role": "user",
#             "content": f"""Context passages:
# {finalChunk}

# Question: {question}

# Answer:"""
#         }
#     ]

#     text_promt = decoder_tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
#     print("\ntext_promt: ", text_promt)
#     input_ids_decoder = decoder_tokenizer.encode(text_promt, return_tensors='pt').to(device)
#     attention_mask = torch.ones_like(input_ids_decoder).to(device)
#     GetFinalAnswer(input_ids_decoder=input_ids_decoder, attention_mask=attention_mask)

In [ ]:
def ManageExtractionPrompt(finalChunk):

    messages = [
        {
            "role": "system",
            "content": """
You are a biomedical findings extraction system.

Extract ONLY explicit biomedical findings and relationships stated in the CONTEXT.

Do NOT extract isolated entity names.

Focus on:
- biological relationships
- molecular mechanisms
- mutations
- biomarker associations
- epigenetic findings
- disease associations

Preserve biomedical entities exactly as written.

Return concise factual bullet points.

Do not infer unstated conclusions.
Do not introduce outside biomedical knowledge.
"""
        },

        {
            "role": "user",
            "content": f"""
CONTEXT:
{finalChunk}

EXTRACTED FINDINGS:
"""
        }
    ]

    text_prompt = decoder_tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    input_ids_decoder = decoder_tokenizer.encode(
        text_prompt,
        return_tensors='pt'
    ).to(device)

    attention_mask = torch.ones_like(input_ids_decoder).to(device)

    return input_ids_decoder, attention_mask

In [ ]:
#Stage 2
def ManageAnswerPrompt(question, extractedFindings):

    messages = [
        {
            "role": "system",
            "content": """
You are a biomedical question answering system.

Answer the QUESTION using ONLY the provided FINDINGS.

Reuse biomedical entities exactly as written.

Do not introduce:
- new mutations
- new biomarkers
- new pathways
- new mechanisms

Do not infer unstated biological consequences.

Return a concise factual answer.
"""
        },

        {
            "role": "user",
            "content": f"""
QUESTION:
{question}

FINDINGS:
{extractedFindings}

ANSWER:
"""
        }
    ]

    text_prompt = decoder_tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    print("\nAnswer Prompt:\n", text_prompt)

    input_ids_decoder = decoder_tokenizer.encode(
        text_prompt,
        return_tensors='pt'
    ).to(device)

    attention_mask = torch.ones_like(input_ids_decoder).to(device)

    return input_ids_decoder, attention_mask

In [ ]:
# # ===== STAGE 1 =====
# # Biomedical extraction

# input_ids_decoder, attention_mask = ManageExtractionPrompt(
#     finalChunk=final_context
# )

# extracted_findings = GetFinalAnswer(
#     input_ids_decoder=input_ids_decoder,
#     attention_mask=attention_mask
# )

# print("\n===== EXTRACTED FINDINGS =====\n")
# print(extracted_findings)


# # ===== STAGE 2 =====
# # Final grounded answer

# input_ids_decoder, attention_mask = ManageAnswerPrompt(
#     question=question,
#     extractedFindings=extracted_findings
# )

# final_answer = GetFinalAnswer(
#     input_ids_decoder=input_ids_decoder,
#     attention_mask=attention_mask
# )

# print("\n===== FINAL ANSWER =====\n")
# print(final_answer)

In [ ]:
def rerankedForm(question,retrieval_chunks,encoder_scores):
  pairs = []
  all_chunks = [i['chunk_text'] for i in retrieval_chunks]
  originalids = [i['original_passage_id'] for i in retrieval_chunks]
  for chunk in all_chunks:
    pairs.append([question,chunk])

  Reranking_tokenize_query = Reranking_token_Model(text=pairs,max_length=512,padding=True,truncation=True,return_tensors='pt')

  with torch.no_grad():
    with torch.amp.autocast('cuda'):
      reranking_model_output = rerank_model(Reranking_tokenize_query['input_ids'],Reranking_tokenize_query['attention_mask'])
      rerank_scores = reranking_model_output.logits.squeeze()

  # normalize encoder scores for top_k results
  encoder_scores = top_scores[0]  # scores from your similarity computation
  encoder_norm = (encoder_scores - encoder_scores.min()) / (encoder_scores.max() - encoder_scores.min())

  # normalize reranker scores
  rerank_norm = (rerank_scores - rerank_scores.min()) / (rerank_scores.max() - rerank_scores.min())

  # combine both
  final_scores = encoder_scores#rerank_scores#0.4 * encoder_norm + 0.6 * rerank_norm
  sorted_indics = torch.argsort(final_scores, descending=True)
  # print('\n\nSorted_indics:',sorted_indics)
  print("\n===== RERANKED RESULTS =====\n")

  #Rerank the chunks again
  RerankedIds = []
  for idx in sorted_indics[:10]:
    RerankedIds.append(originalids[idx])
    # print(f"Rank {i+1}")
    # print(f"Rerank Score: {rerank_scores[idx].item():.4f}")
    # print("-" * 50)
    # print(retrieval_chunks[idx])
    # print("\n")
  final_chunks = [
    all_chunks[idx]
    for idx in sorted_indics[:5]
  ]
  print('\nRerankedIDS: ',RerankedIds)
  EvaluationData["retrieved_passage_ids"] = RerankedIds
  final_context = "\n\n".join(final_chunks)
  EvaluationData["retrieved_context"] = final_context
  # ===== STAGE 1 =====
  # Extraction

  input_ids_decoder, attention_mask = ManageExtractionPrompt(
    finalChunk=final_context
  )

  extracted_findings = GetFinalAnswer(
    input_ids_decoder=input_ids_decoder,
    attention_mask=attention_mask
  )

  print("\n===== EXTRACTED FINDINGS =====\n")
  print(extracted_findings)


  # ===== STAGE 2 =====
  # Final grounded answer

  input_ids_decoder, attention_mask = ManageAnswerPrompt(
    question=question,
    extractedFindings=extracted_findings
  )

  final_answer = GetFinalAnswer(
    input_ids_decoder=input_ids_decoder,
    attention_mask=attention_mask
  )

  print("\n===== FINAL ANSWER =====\n")
  print(final_answer)
  EvaluationData['predicted_answer'] = final_answer
  results = evaluate_rag(EvaluationData)
  EvaluationData['metrics'] = results
  print(results)
  # ManageDecoderModel(finalChunk=final_context,question=question)

In [ ]:
for quesIdx,i in enumerate(train_quesAnsPassage['question'][0:1],start=0):
  retrieval_chunks = []
  #Tokenize
  queryQ = i
  query_tokenize = query_tokenizer(text=queryQ,padding='max_length',max_length=64,add_special_tokens=True,truncation=True,return_attention_mask=True,return_tensors='pt').to(device)

  #Embedding
  with torch.no_grad():
    with torch.amp.autocast('cuda'):
      query_model_output = query_encoder(query_tokenize['input_ids'],query_tokenize['attention_mask'])
  query_vector_cpu = query_model_output.cpu()

  #matrix multiplaction
  vector_database_normalize = F.normalize(vector_database,p=2,dim=-1)
  vector_query_normalize = F.normalize(query_vector_cpu,p=2,dim=-1)

  similarity_score = vector_query_normalize @ vector_database_normalize.T
  # ---- DIAGNOSTIC: ADD HERE ----
  target_ids = [23179372, 19270706, 23184418]
  all_scores_sorted = similarity_score[0].argsort(descending=True)  # sort all chunks

  for rank, idx in enumerate(all_scores_sorted):
    chunk_id = knowledge_strings[idx]['original_passage_id']
    if chunk_id in target_ids:
        print(f"Rank {rank} | score={similarity_score[0][idx]:.4f} | passage_id={chunk_id}")
  # ---- END DIAGNOSTIC ----
  #Get top 3 scores:
  top_k=15
  top_scores,top_indics = torch.topk(input=similarity_score,k=top_k,dim=-1)
  print(f'======Matching=====')
  print(f'top_scores: {top_scores} | top_indexes: {top_indics}')

  print(f'========'*50)
  print(f'\n\nQuestion:{quesIdx}: {i}')
  print(f'\n\nExpected Answer: {train_quesAnsPassage['answer'][quesIdx]}')
  print(f'\n\nExpected AnswerID: {train_quesAnsPassage['relevant_passage_ids'][quesIdx]}')
  EvaluationData["question"] = i
  EvaluationData["Expected_Answer"] = train_quesAnsPassage['answer'][quesIdx]
  EvaluationData["Expected_Answer_ID"] = train_quesAnsPassage['relevant_passage_ids'][quesIdx]
  savedids = []
  for idx in top_indics[0]: # here put 0 because top_indics has 2D array [[1,2,3,4,5]] kinda
   # print(f'\nRetrived-AnswerID: {knowledge_strings[idx]['original_passage_id']}')
   retrieval_chunks.append(knowledge_strings[idx])
   savedids.append(knowledge_strings[idx]['original_passage_id'])
  print("Retrive-IDS: ",savedids)
  rerankedForm(question=i,retrieval_chunks=retrieval_chunks,encoder_scores=top_scores[0].cpu())

Rank 0 | score=0.6278 | passage_id=19270706
Rank 1 | score=0.6224 | passage_id=23179372
Rank 4 | score=0.6191 | passage_id=19270706
Rank 6 | score=0.6124 | passage_id=23184418
Rank 13 | score=0.6043 | passage_id=19270706
Rank 25 | score=0.5910 | passage_id=23179372
Rank 32 | score=0.5850 | passage_id=23184418
Rank 53 | score=0.5742 | passage_id=23184418
Rank 89 | score=0.5600 | passage_id=23184418
Rank 91 | score=0.5592 | passage_id=23179372
Rank 676 | score=0.5184 | passage_id=23179372
======Matching=====
top_scores: tensor([[0.6278, 0.6224, 0.6204, 0.6197, 0.6191, 0.6135, 0.6124, 0.6113, 0.6092,
         0.6085, 0.6075, 0.6071, 0.6051, 0.6043, 0.6021]]) | top_indexes: tensor([[ 41168,  72430, 107373,  78769,  41169,  91639,  72502,  56739, 114113,
          77608, 103132,  81249,  84369,  41167,  37333]])


Question:0: What is the implication of histone lysine methylation in medulloblastoma?


Expected Answer: Aberrant patterns of H3K4, H3K9, and H3K27 histone lysine methylation were

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-large
Key                             | Status     | 
--------------------------------+------------+-
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
pooler.dense.weight             | MISSING    | 
pooler.dense.bias               | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


{'hit@5': 1, 'recall@5': 1.0, 'precision@5': 0.6, 'mrr': 1.0, 'ndcg@5': np.float64(0.8584912361856167), 'rouge': {'rouge1': 0.449438202247191, 'rougeL': 0.24719101123595502}, 'bertscore': {'precision': 0.8783791661262512, 'recall': 0.9110771417617798, 'f1': 0.8944293856620789}, 'semantic_similarity': np.float32(0.9026197), 'context_precision': np.float32(0.8753527), 'context_recall': np.float32(0.6157894), 'faithfulness': 0.9330658316612244, 'hallucination_score': 0.06693416833877563, 'answer_relevance': np.float32(0.78599536)}


In [ ]:
print("\n===== EVALUATION RESULTS =====\n")

for key, value in EvaluationData.items():

    if key != "metrics":

        print(f"{key} :\n{value}\n")

print("\n===== METRICS =====\n")

for metric_name, metric_value in EvaluationData['metrics'].items():

    print(f"{metric_name} : {metric_value}")


===== EVALUATION RESULTS =====

question :
What is the implication of histone lysine methylation in medulloblastoma?

Expected_Answer :
Aberrant patterns of H3K4, H3K9, and H3K27 histone lysine methylation were shown to result in histone code alterations, which induce changes in gene expression, and affect the proliferation rate of cells in medulloblastoma.

Expected_Answer_ID :
[23179372, 19270706, 23184418]

retrieved_passage_ids :
[19270706, 23179372, 27864305, 23715325, 19270706]

retrieved_context :
Notably, we identified previously unknown amplifications and 
homozygous deletions, including recurrent, mutually exclusive, highly focal 
genetic events in genes targeting histone lysine methylation, particularly that 
of histone 3, lysine 9 (H3K9). Post-translational modification of histone 
proteins is critical for regulation of gene expression, can participate in 
determination of stem cell fates and has been implicated in carcinogenesis. Consistent with our genetic data, restorat

In [ ]:
# find and print the best chunk from this passage
for chunk in knowledge_strings:
    if chunk['original_passage_id'] == 23184418:
        print(chunk['chunk_text'])
        break

> Evaluation Metrics

In [ ]:
def evaluate_rag(EvaluationData):

    results = {}

    # =========================
    # RETRIEVAL METRICS
    # =========================

    results['hit@5'] = hit_at_k(
        EvaluationData['Expected_Answer_ID'],
        EvaluationData['retrieved_passage_ids'],
        k=5
    )

    results['recall@5'] = recall_at_k(
        EvaluationData['Expected_Answer_ID'],
        EvaluationData['retrieved_passage_ids'],
        k=5
    )

    results['precision@5'] = precision_at_k(
        EvaluationData['Expected_Answer_ID'],
        EvaluationData['retrieved_passage_ids'],
        k=5
    )

    results['mrr'] = reciprocal_rank(
        EvaluationData['Expected_Answer_ID'],
        EvaluationData['retrieved_passage_ids']
    )

    results['ndcg@5'] = calculate_ndcg(
        EvaluationData['Expected_Answer_ID'],
        EvaluationData['retrieved_passage_ids'],
        k=5
    )

    # =========================
    # GENERATION METRICS
    # =========================

    results['rouge'] = rouge_metric(
        EvaluationData['Expected_Answer'],
        EvaluationData['predicted_answer']
    )

    results['bertscore'] = bertscore_metric(
        EvaluationData['Expected_Answer'],
        EvaluationData['predicted_answer']
    )

    results['semantic_similarity'] = semantic_similarity(
        EvaluationData['Expected_Answer'],
        EvaluationData['predicted_answer']
    )

    # =========================
    # GROUNDEDNESS METRICS
    # =========================

    results['context_precision'] = context_precision(
        EvaluationData['retrieved_context'],
        EvaluationData['predicted_answer']
    )

    results['context_recall'] = context_recall(
        EvaluationData['retrieved_context'],
        EvaluationData['predicted_answer']
    )

    results['faithfulness'] = faithfulness(
        EvaluationData['retrieved_context'],
        EvaluationData['predicted_answer']
    )

    results['hallucination_score'] = hallucination_score(
        EvaluationData['retrieved_context'],
        EvaluationData['predicted_answer']
    )

    results['answer_relevance'] = answer_relevance(
        EvaluationData['question'],
        EvaluationData['predicted_answer']
    )

    return results

In [ ]:
!pip install sentence-transformers rouge-score bert-score
from transformers import pipeline
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from bert_score import score
from rouge_score import rouge_scorer
from sklearn.metrics import ndcg_score

sentence_Transformer_Model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
nli_pipeline = pipeline("text-classification",model="MoritzLaurer/deberta-v3-large-zeroshot-v2.0")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 2.0 MB/s eta 0:00:00


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/23.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/970 [00:00<?, ?B/s]

In [ ]:
#Hit@k: Did we retrive at least one correct document?
def hit_at_k(gold_ids, retrieved_ids, k=5):
    top_k = retrieved_ids[:k]
    for doc_id in top_k:
        if doc_id in gold_ids:
            return 1
    return 0

In [ ]:
#Recall@K: How many relevent documents were retrieved?
def recall_at_k(gold_ids, retrieved_ids, k=5):
    top_k = retrieved_ids[:k]
    relevant = 0
    for doc_id in top_k:
        if doc_id in gold_ids:
            relevant += 1
    return relevant / len(gold_ids)

In [ ]:
#Precision@K: How many retrieved documents are actually correct?
def precision_at_k(gold_ids, retrieved_ids, k=5):
    top_k = retrieved_ids[:k]
    relevant = 0
    for doc_id in top_k:
        if doc_id in gold_ids:
            relevant += 1
    return relevant / k

In [ ]:
#MRR (Mean Reciprocal Rank): How early was the FIRST correct document retrieved?
def reciprocal_rank(gold_ids, retrieved_ids):
    for rank, doc_id in enumerate(retrieved_ids, start=1):
        if doc_id in gold_ids:
            return 1 / rank
    return 0

In [ ]:
#nDCG: Measures ranking quality.
def calculate_ndcg(gold_ids, retrieved_ids, k=5):
    relevance = []
    for doc_id in retrieved_ids[:k]:
        if doc_id in gold_ids:
            relevance.append(1)
        else:
            relevance.append(0)
    ideal = sorted(relevance, reverse=True)
    return ndcg_score([ideal], [relevance])

GENERATION METRICS

In [ ]:
#ROUGE
def rouge_metric(reference, prediction):
    scorer = rouge_scorer.RougeScorer(
        ['rouge1', 'rougeL'],
        use_stemmer=True
    )
    scores = scorer.score(reference, prediction)
    return {
        "rouge1": scores['rouge1'].fmeasure,
        "rougeL": scores['rougeL'].fmeasure
    }

In [ ]:
#BERTScore
def bertscore_metric(reference, prediction):
    P, R, F1 = score(
        [prediction],
        [reference],
        lang='en'
    )
    return {
        "precision": P.mean().item(),
        "recall": R.mean().item(),
        "f1": F1.mean().item()
    }

In [ ]:
#Semantic Similarity
def semantic_similarity(reference, prediction):
    embeddings = sentence_Transformer_Model.encode(
        [reference, prediction]
    )
    similarity = cosine_similarity(
        [embeddings[0]],
        [embeddings[1]]
    )
    return similarity[0][0]

GROUNDEDNESS METRICS

In [ ]:
#Context Precision: Did the answer actually use retrieved context?
def context_precision(retrieved_context,
                      predicted_answer):
    embeddings = sentence_Transformer_Model.encode(
        [retrieved_context, predicted_answer]
    )
    similarity = cosine_similarity(
        [embeddings[0]],
        [embeddings[1]]
    )
    return similarity[0][0]

In [ ]:
#Context Recall: Did the answer cover important retrieved facts?
def context_recall(retrieved_context,
                   predicted_answer):

    # split retrieved context into sentences
    context_sentences = [
        sent.strip()
        for sent in retrieved_context.split('.')
        if sent.strip()
    ]

    # embed predicted answer once
    answer_embedding = sentence_Transformer_Model.encode(
        [predicted_answer]
    )

    similarities = []

    # compare every context sentence against answer
    for sentence in context_sentences:

        sentence_embedding = sentence_Transformer_Model.encode(
            [sentence]
        )

        similarity = cosine_similarity(
            sentence_embedding,
            answer_embedding
        )[0][0]

        similarities.append(similarity)

    # average coverage score
    return np.mean(similarities)

In [ ]:
#Faithfulness (NLI-Based): measures hallucination
def faithfulness(retrieved_context,predicted_answer):
    result = nli_pipeline( f"{retrieved_context} [SEP] {predicted_answer}")
    label = result[0]['label'].lower()
    score = result[0]['score']
    if label == "entailment":
        return score
    elif label == "contradiction":
        return 1 - score
    else:
        return 0.5

In [ ]:
#Hallucination Score
def hallucination_score(retrieved_context,predicted_answer):
    faithful_score = faithfulness(retrieved_context,predicted_answer)
    return 1 - faithful_score

In [ ]:
#Answer Relevance
def answer_relevance(question,predicted_answer):
    embeddings = sentence_Transformer_Model.encode([question, predicted_answer])
    similarity = cosine_similarity([embeddings[0]],[embeddings[1]])
    return similarity[0][0]